# Contextual Multi-Armed Bandit

For the contextual multi-armed bandit (cMAB) when user information is available (context), we implemented a generalisation of Thompson sampling algorithm ([Agrawal and Goyal, 2014](https://arxiv.org/pdf/1209.3352.pdf)) based on NumPyro.

![title](img/cmab.png)

The following notebook contains an example of usage of the class Cmab, which implements the algorithm above.

In [1]:
import numpy as np

from pybandits.cmab import CmabBernoulli
from pybandits.model import BayesianNeuralNetwork, BnnLayerParams, BnnParams, FeaturesConfig, StudentTArray

/home/runner/.cache/pypoetry/virtualenvs/pybandits-vYJB-miV-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
n_samples = 1000
n_features = 5

First, we need to define the input context matrix $X$ of size ($n\_samples, n\_features$) and the mapping of possible actions $a_i \in A$ to their associated model.

In [3]:
# context
X = 2 * np.random.random_sample((n_samples, n_features)) - 1  # random float in the interval (-1, 1)
print("X: context matrix of shape (n_samples, n_features)")
print(X[:10])

X: context matrix of shape (n_samples, n_features)
[[-0.84902284  0.73533697  0.11449077  0.27326443 -0.85327431]
 [-0.68876913  0.507391   -0.26447278  0.33487447 -0.60533834]
 [-0.45558024  0.72171067 -0.76540726  0.81322327  0.64394703]
 [ 0.40792383  0.44783707 -0.67687233  0.35037202 -0.05752822]
 [ 0.78812978  0.33303642 -0.91643312 -0.34801155 -0.88680455]
 [ 0.49107852 -0.71513548  0.24090308  0.17019137 -0.31519956]
 [-0.48444562  0.84287714  0.11720863  0.15438249 -0.63520841]
 [-0.97482046 -0.29286474 -0.76114254 -0.19698918 -0.16855604]
 [-0.98761174 -0.92192165  0.93769083  0.53491842 -0.34187264]
 [-0.06489629  0.07786136 -0.26758214 -0.40542587  0.26388793]]


In [4]:
# define action model
bias = StudentTArray.cold_start(mu=1, sigma=2, shape=1)
weight = StudentTArray.cold_start(shape=(n_features, 1))
layer_params = BnnLayerParams(weight=weight, bias=bias)
model_params = BnnParams(bnn_layer_params=[layer_params])
feature_config = FeaturesConfig(n_features=n_features)

update_method = "VI"
update_kwargs = {"num_steps": 100, "batch_size": 128, "optimizer_type": "adam"}

actions = {
    "a1": BayesianNeuralNetwork(
        model_params=model_params,
        feature_config=feature_config,
        update_method=update_method,
        update_kwargs=update_kwargs,
    ),
    "a2": BayesianNeuralNetwork(
        model_params=model_params,
        feature_config=feature_config,
        update_method=update_method,
        update_kwargs=update_kwargs,
    ),
}

We can now init the bandit given the mapping of actions $a_i$ to their model.

In [5]:
# init contextual Multi-Armed Bandit model
cmab = CmabBernoulli(actions=actions)

The predict function below returns the action selected by the bandit at time $t$: $a_t = argmax_k P(r=1|\beta_k, x_t)$. The bandit selects one action per each sample of the contect matrix $X$.

In [6]:
# predict action
pred_actions, _, _ = cmab.predict(X)
print("Recommended action: {}".format(pred_actions[:10]))

Recommended action: ['a1', 'a2', 'a1', 'a1', 'a1', 'a1', 'a1', 'a1', 'a2', 'a2']


Now, we observe the rewards and the context from the environment. In this example rewards and the context are randomly simulated.

In [7]:
# simulate reward from environment
simulated_rewards = np.random.randint(2, size=n_samples).tolist()
print("Simulated rewards: {}".format(simulated_rewards[:10]))

Simulated rewards: [1, 0, 0, 0, 0, 1, 0, 1, 0, 1]


Finally, we update the model providing per each action sample: (i) its context $x_t$ (ii) the action $a_t$ selected by the bandit, (iii) the corresponding reward $r_t$.

In [8]:
# update model
cmab.update(context=X, actions=pred_actions, rewards=simulated_rewards)

SVI:   0%|          | 0/25 [00:00<?, ?it/s]

SVI:   4%|▍         | 1/25 [00:01<00:24,  1.04s/it]

SVI:   4%|▍         | 1/25 [00:01<00:24,  1.04s/it, loss=768.6033]

SVI:   8%|▊         | 2/25 [00:01<00:23,  1.04s/it, loss=830.1647]

SVI:  12%|█▏        | 3/25 [00:01<00:22,  1.04s/it, loss=828.9351]

...

SVI:  91%|█████████ | 31/34 [00:00<00:02,  1.13it/s, loss=373.8156]

SVI:  94%|█████████▍| 32/34 [00:00<00:01,  1.13it/s, loss=377.6057]

SVI:  97%|█████████▋| 33/34 [00:00<00:00,  1.13it/s, loss=375.2412]

SVI: 100%|██████████| 34/34 [00:01<00:00, 22.99it/s, loss=375.2412]

SVI: 100%|██████████| 34/34 [00:01<00:00, 22.99it/s, loss=395.3547]